<a href="https://colab.research.google.com/github/sreekanthTa/Pii_finetuning/blob/main/FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers datasets trl peft accelerate bitsandbytes

In [ ]:
!nvidia-smi

Tue Sep 15 03:49:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from datasets import load_dataset, DatasetDict, Features, Sequence, Value

data_files = {
    "train": "/content/sample_data/fine/train.jsonl",
    "validation": "/content/sample_data/fine/validation.jsonl",
    "test": "/content/sample_data/fine/test.jsonl",
}

dataset = load_dataset("json", data_files=data_files)

dataset

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 200
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 200
    })
})

In [ ]:
#Convert to train format of data



In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
quantize_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # Changed from bfloat16 to float16 for T4 compatibility
)

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantize_config,
    device_map="auto",
    dtype=torch.float16 # Changed from torch_dtype to dtype as it's deprecated
)
# Removed explicit model.to(torch.float16) as it caused memory issues and is often redundant with dtype/bnb_4bit_compute_dtype settings

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
print("Model loaded successfully!")
print("Device:", model.device)

Model loaded successfully!
Device: cuda:0


In [ ]:
print(f"Model dtype: {model.dtype}")
if hasattr(model, 'parameters') and next(model.parameters(), None) is not None:
    print(f"First parameter dtype: {next(model.parameters()).dtype}")
else:
    print("Model does not have accessible parameters.")

Model dtype: torch.float16
First parameter dtype: torch.float16


In [ ]:
from peft import LoraConfig, TaskType

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    task_type=TaskType.CAUSAL_LM,
    bias="none"
)

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-pii-model",

    # Train with very small batches because T4 has limited VRAM
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    # Accumulate 8 batches before updating the model
    gradient_accumulation_steps=8,

    # Number of times to go through the training data (reduced to 1 to speed up training)
    num_train_epochs=1,

    # Learning rate for LoRA
    learning_rate=2e-4,

    # Save/check the model after each epoch
    save_strategy="epoch",

    # Evaluate after each epoch
    eval_strategy="epoch",

    # Print training information
    logging_steps=10,

    # Use FP16 because we're on a T4
    fp16=True,
    bf16=False,

    # Helps reduce GPU memory usage (temporarily disabled to diagnose BFloat16 error)
    gradient_checkpointing=False,

    # Maximum sequence length
    max_length=512,

    # Don't use an external experiment tracker
    report_to="none"
)

In [ ]:
from trl import SFTTrainer
import torch

# Re-initialize trainer to apply the updated training_args (1 epoch)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
    peft_config=lora_config,
    processing_class=tokenizer,
)

# Ensure all trainable parameters are strictly float32
# This prevents BFloat16 tensors from reaching the GradScaler on the T4 GPU
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

trainer.train()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.177835,0.186196,0.188015,437595.000000,0.939880
2,0.184458,0.186001,0.182979,875190.000000,0.940483


In [ ]:
# Folder where we will save the trained LoRA adapter
OUTPUT_DIR = "./qwen-pii-model"

# Save the trained adapter
trainer.save_model(OUTPUT_DIR)

# Save the tokenizer as well
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved successfully!")